In [2]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
print('LLM ready')

LLM ready


In [3]:
GOOD_PROMPT = ChatPromptTemplate.from_template("""
You are BajajBot, an AI assistant for Bajaj Finance helpdesk agents.

CONTEXT (retrieved from Bajaj Finance policy documents):
-------------------------------------------------------
{context}
-------------------------------------------------------

INSTRUCTIONS:
- Answer ONLY using the CONTEXT above.
- Do NOT use your training knowledge.
- If the answer is not in the CONTEXT, say exactly:
  "I don't have this information in the provided documents."
- Keep your answer under 3 sentences.
- If a specific number or rule is in CONTEXT, include it.

QUESTION: {question}

ANSWER:
""")

In [4]:
chain = GOOD_PROMPT | llm | StrOutputParser()


In [8]:
# pip install langchain-community

In [9]:
# pip install pypdf

In [10]:
PDF_PATH = 'data/bajaj_finance_policy_reference.pdf'

from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader(PDF_PATH)
pages  = loader.load()

In [12]:
len(pages)

12

In [16]:
pages[0].page_content.split('\n')

['BAJAJ FINANCE LIMITED',
 'Internal Policy Reference Document — CONFIDENTIAL',
 'Page 1',
 'For internal use only | Bajaj Finance Limited | FY 2024-25 | Unauthorised distribution prohibited',
 ' BAJAJ FINANCE LIMITED',
 ' Policy & Procedure Reference',
 ' Document',
 ' Helpdesk Agent Knowledge Base',
 ' FY 2024–25 | All Loan Products',
 'Document Type',
 'Internal Policy & Procedure Manual',
 'Coverage',
 'Personal Loan · Home Loan · Gold Loan · Business Loan · CIBIL Policy',
 'Intended Users',
 'Helpdesk Agents · Branch Executives · Collections Team · Sales Team',
 'Classification',
 'CONFIDENTIAL — For Internal Use Only',
 'Version',
 'v3.2 — Revised April 2024',
 'Approved by',
 'VP Operations & Chief Risk Officer',
 'Next Review',
 'October 2024',
 'Document Sections',
 ' #',
 ' Section',
 ' Ref',
 '01',
 ' Personal Loan — Eligibility & Rates',
 ' Page 2',
 '02',
 ' Personal Loan — Required Documents',
 ' Page 3',
 '03',
 ' Home Loan — Eligibility & Terms',
 ' Page 4',
 '04',
 ' L

In [ ]:
# next big question
len(pages[0].page_content)

# splitting == chunks ??


1153

In [21]:
SAMPLE_TEXT="""GOLD LOAN POLICY — BAJAJ FINANCE

Gold purity accepted: 18 karat to 24 karat.
Maximum LTV: 75 percent of market value (RBI mandated).
Tenure: 3 months to 24 months.

Foreclosure Policy:
Within 3 months: 2 percent foreclosure charge
3 to 6 months: 2 percent charge
6 to 12 months: 1 percent charge
After 12 months: No foreclosure charge

Disbursal: Same day within 30 minutes of appraisal.
"""


In [23]:
chunk_sixe = 200
chunks = [SAMPLE_TEXT[i:i+chunk_sixe] for i in range(0, len(SAMPLE_TEXT), chunk_sixe)]
chunks[0]

'GOLD LOAN POLICY — BAJAJ FINANCE\n\nGold purity accepted: 18 karat to 24 karat.\nMaximum LTV: 75 percent of market value (RBI mandated).\nTenure: 3 months to 24 months.\n\nForeclosure Policy:\nWithin 3 month'

In [24]:
chunks[1]

's: 2 percent foreclosure charge\n3 to 6 months: 2 percent charge\n6 to 12 months: 1 percent charge\nAfter 12 months: No foreclosure charge\n\nDisbursal: Same day within 30 minutes of appraisal.\n'

In [26]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create a text splitter
# chunk_size = maximum size of one chunk
# chunk_overlap = small repeated part between two nearby chunks
# separators = order in which LangChain tries to split the text
splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=64,
    separators=["\n\n", "\n", ". ", " ", ""],
    length_function=len, #length_function=len tells the splitter how to measure the size of text
)

chunks = splitter.split_documents(pages)

In [30]:
len(chunks[0].page_content)

512

In [31]:
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 28058.28it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [33]:
chunk_texts = [c.page_content for c in chunks]
chunk_meta  = [c.metadata     for c in chunks]

chunk_vectors = model.encode(
    chunk_texts,
    show_progress_bar = True,
    convert_to_numpy  = True,
    normalize_embeddings = True,   # makes cosine == dot product (faster)
)



Batches: 100%|██████████| 2/2 [00:02<00:00,  1.20s/it]


In [35]:
# chunk_vectors

In [36]:
store = {
    'vectors' : chunk_vectors,
    'texts'   : chunk_texts,
    'meta'    : chunk_meta,
    'model'   : 'all-MiniLM-L6-v2',   # CRITICAL: remember which model produced these
}

import pickle

with open('bajaj_vectors.pkl', 'wb') as f:
    pickle.dump(store, f)

In [37]:
with open('bajaj_vectors.pkl', 'rb') as f:
    loaded = pickle.load(f)

In [39]:
from sklearn.metrics.pairwise import cosine_similarity


def search_similarity(query, top_k=3):
    # Convert the query into an embedding
    query_vector = model.encode([query], convert_to_numpy=True)

    # Compare query with all chunk vectors using cosine similarity
    scores = cosine_similarity(query_vector, chunk_vectors)[0]

    # Get indexes of top matching chunks
    top_indexes = scores.argsort()[::-1][:top_k]

    # Return (score, index)
    results = []
    for i in top_indexes:
        results.append((float(scores[i]), int(i)))

    return results



q_relevant = "What is the foreclosure charge on gold loans?"
q_irrelevant = "How do I bake a chocolate cake?"


In [40]:
def show(title, results, query):
    print("\n===", title, "===")
    print("Query:", query)
    print("Found:", len(results))

    for i, (score, idx) in enumerate(results, 1):
        text = chunk_texts[idx][:110].replace("\n", " ")
        page = chunk_meta[idx].get("page", "?")
        print(f"{i}. score={score:.4f} | page {page} | {text}...")

In [41]:
show("Similarity (relevant)", search_similarity(q_relevant), q_relevant)
show("Similarity (IRRELEVANT)", search_similarity(q_irrelevant), q_irrelevant)



=== Similarity (relevant) ===
Query: What is the foreclosure charge on gold loans?
Found: 3
1. score=0.7085 | page 8 | 7.2 Foreclosure Policy Gold loan foreclosure (early closure before tenure end) is governed by the following po...
2. score=0.6014 | page 8 | Maximum 75% of gold market value (RBI mandated ceiling) Tenure 3 months to 24 months | Bullet repayment or mon...
3. score=0.4401 | page 1 | ■1,000 + GST per bounced NACH/ECS mandate Prepayment (Full) 4% of outstanding principal if closed within 12 mo...

=== Similarity (IRRELEVANT) ===
Query: How do I bake a chocolate cake?
Found: 3
1. score=0.1027 | page 1 | Age 21 – 60 years for salaried | 25 – 65 years for self-employed Minimum Income ■25,000/month net take-home (s...
2. score=0.0974 | page 7 | 14% – 24% p.a. based on CIBIL and business vintage Collateral Not required for amounts up to ■50 lakhs | Requi...
3. score=0.0506 | page 0 | Page 7 07  Gold Loan — Products & Process  Page 8 08  CIBIL Score — Impact & Reporting  Page 9

In [42]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

GOOD_PROMPT = ChatPromptTemplate.from_template("""
You are BajajBot, an AI assistant for Bajaj Finance helpdesk agents.

CONTEXT (retrieved from Bajaj Finance policy documents):
-------------------------------------------------------
{context}
-------------------------------------------------------

INSTRUCTIONS:
- Answer ONLY using the CONTEXT above.
- Do NOT use your training knowledge.
- If the answer is not in the CONTEXT, say exactly:
  "I don't have this information in the provided documents."
- Keep your answer under 3 sentences.
- If a specific number or rule is in CONTEXT, include it.

QUESTION: {question}

ANSWER:
""")

In [43]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

chain = GOOD_PROMPT | llm | StrOutputParser()

In [49]:

def search_similarity(query, top_k=3):
    query_vector = model.encode([query], convert_to_numpy=True)
    scores = cosine_similarity(query_vector, chunk_vectors)[0]
    top_indexes = scores.argsort()[::-1][:top_k]

    results = []
    for i in top_indexes:
        results.append((float(scores[i]), int(i)))

    return results


def get_context(query, top_k=3):
    results = search_similarity(query, top_k=top_k)
    print(results)

    context_parts = []
    for score, idx in results:
        text = chunk_texts[idx]
        context_parts.append(text)

    print(context_parts)

    context = "\n\n".join(context_parts)
    return context

In [52]:
query = "What is the foreclosure charge on gold loans?"

context = get_context(query, top_k=3)

print("Context:\n")


answer = chain.invoke({
    "context": context,
    "question": query
})

print("\nAnswer:\n")
print(answer)

[(0.7085356712341309, 29), (0.6013743877410889, 28), (0.4401366114616394, 7)]
['7.2 Foreclosure Policy\nGold loan foreclosure (early closure before tenure end) is governed by the following policy. Foreclosure is the most\ncommon query from gold loan customers.\nWithin 3 months of\ndisbursal\n2% foreclosure charge on outstanding principal + accrued interest\n3 – 6 months\n2% foreclosure charge on outstanding principal\n6 – 12 months\n1% foreclosure charge on outstanding principal\nAfter 12 months\nNO foreclosure charge | Customer pays only outstanding principal + interest\nClosure Process', 'Maximum 75% of gold market value (RBI mandated ceiling)\nTenure\n3 months to 24 months | Bullet repayment or monthly interest options\nInterest Rate\n9.5% – 18% p.a. | 9.5%–12% for high-value (above ■5 lakhs) | Monthly interest option available\nAppraisal Process\nIn-branch gold testing using XRF machine | Results in 15 minutes\nDisbursal Time\nSame day — within 30 minutes of appraisal and KYC compl